In [26]:
from dotenv import load_dotenv
import os

load_dotenv()

def grid_to_row_strings(grid):
    return [' '.join(map(str, row)) for row in grid]

PROMPT = """
{input_output_pairs}
Given this group of input/output 
{code}
And the following solver code 
"Cheat" and output a reasoning that would produce this code given the problem, without quoting the code 
As if You're given the Input/Output
And your goal is to get the reasoning that leads to the code
Output the reasoning that you would write to get to the code, like "Here's how I would solve it"
Output:"""

In [ ]:
import tqdm
from openai import OpenAI
from loader import load_training_problem, list_training_problems

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

problems = list_training_problems()
for problem_id in tqdm.tqdm(problems):
    solver_path = f"remapped_solvers/{problem_id}.py"
    with open(solver_path, 'r') as f:
        solver_code = f.read()

    problem = load_training_problem(problem_id)

    input_output_pairs = ""
    for elem in problem['train']:
        pb_input ="\n".join(grid_to_row_strings(elem['input']))
        pb_output = "\n".join(grid_to_row_strings(elem['output']))
        input_output_pairs += f"Input:\n{pb_input}\nOutput:\n{pb_output}\n\n"

    response = client.responses.create(
        model="gpt-5",
        input=PROMPT.format(input_output_pairs=input_output_pairs, code=solver_code),
        reasoning={ "effort": "medium" },
        text={ "verbosity": "medium" },
    )

    with open(f"reasoning_files/{problem_id}.txt", "w", encoding="utf-8") as f:
        f.write(response.output_text)

 88%|████████▊ | 351/400 [3:44:08<30:07, 36.88s/it]  